# Global sea level change time series from 1950 to 2050 derived from reanalysis and high resolution CMIP6 climate projections

Production date: 2026-05-12

**Please note that this repository is used for development and review, so quality assessments should be considered work in progress until they are merged into the main branch.**

Historical & future experiments versions: v1 \
Reanalysis experiment version: v3

Produced by: Enis Gerxhalija, Olivier Burggraaff (National Physical Laboratory).

## 🌍 Use case: Retrieving sea-level return periods from timeseries of reanalysis and high resolution CMIP6 climate projections.

## ❓ Quality assessment question
* **How consistent are the HighResMIP ensembles for the present epoch (1985-2014) against those derived from the ERA5 climate reanalysis?**

**‘Context paragraph’ (no title/heading)** - a very short introduction before the assessment statement describing approach taken to answer the user question. One or two key references could be useful, if the assessment summarises literature.

**Background**
Check documentation: Only one model? Mean?

## 📢 Quality assessment statement

```{admonition} These are the key outcomes of this assessment
:class: note
* Finding 1
* Finding 2
* Finding 3
* etc
```

## 📋 Methodology

A ‘free text’ introduction to the data analysis steps or a description of the literature synthesis, with a justification of the approach taken, and limitations mentioned. **Mention which CDS catalogue entry is used, including a link, and also any other entries used for the assessment**.
[Global sea level change time series from 1950 to 2050 derived from reanalysis and high resolution CMIP6 climate projections](https://doi.org/10.24381/cds.a6d42d60)

**Note:** This notebook is currently just a brain-dump in anticipation of starting the actual quality assessment at a later stage.

E.g. 'The analysis and results are organised in the following steps, which are detailed in the sections below:' 

**[](section-setup)**
 * Import all required libraries.
 * Define helper functions.

**[](section-download)** 
 * Set up CDS downloads for reanalysis and HighResMIP ensemble in present epoch (1985-2014).

**[](section-statistics)**
 * Sub-steps or key points listed in bullet below. No strict requirement to match and link to sub-headings.

**[](section-results)**
 * Sub-steps or key points listed in bullet below. No strict requirement to match and link to sub-headings.

Any further notes on the method could go here (explanations, caveats or limitations).

## 📈 Analysis and results

(section-setup)=
### 1. Code setup

#### Imports

In [1]:
# Input / Output
from pathlib import Path
import earthkit.data as ekd
import warnings

# General data handling
import numpy as np
np.seterr(divide="ignore")  # Ignore divide-by-zero warnings
import pandas as pd
import xarray as xr
from functools import partial
from dask.array.core import PerformanceWarning
warnings.simplefilter(action="ignore", category=PerformanceWarning)
from itertools import batched

# Visualisation
import earthkit.plots as ekp
from earthkit.plots.styles import Style
import matplotlib.pyplot as plt
plt.rcParams["grid.linestyle"] = "--"
from matplotlib.colors import LogNorm
import cmcrameri as cmc
from tqdm import tqdm  # Progress bars

# Visualisation in Jupyter book -- automatically ignored otherwise
try:
    from myst_nb import glue
except ImportError:
    glue = None

#### Helper functions

##### General

In [2]:
# Type hints for helper functions
from typing import Callable, Optional, Iterable

# For pre-defining functions
from functools import partial

##### Downloading data

In [3]:
## Data downloading
def domain_to_request(domain: ekp.geo.domains.Domain) -> dict:
    """ From an earthkit-plots domain, generate a request for earthkit-data / cdsapi. """
    bbox = domain.bbox.to_latlon_bbox()

    # Round
    north = int(np.ceil(bbox.north) + 1)
    south = int(np.floor(bbox.south) - 1)
    west = int(np.floor(bbox.west) - 1)
    east = int(np.ceil(bbox.east) + 1)
    
    area = [north, west, south, east]
    return {"area": area}

# Handling CDS size limits
def batch_requests(main_request: dict, *, batch_key: str="year", n: int=20) -> list[dict]:
    """ Take a big request (e.g. ERA5–Drought for all years) and separate it into smaller ones (size `n`). """
    full_range = main_request[batch_key]  # e.g. [1940, 1941, ..., 2024]
    batched_range = batched(full_range, n)  # e.g. [1940, ..., 1959], [1960, ..., 1979], ...
    subrequests = [main_request | {batch_key: batch} for batch in batched_range]  # create corresponding CDS requests
    return subrequests

##### Data (pre-)processing

In [4]:
## Loops for convenience
def loop_over_(*args, progress=True, **kwargs) -> tqdm:
    """ Generate a tqdm progressbar; inverts `disable` keyword """
    return tqdm(*args, disable=not progress, leave=False, **kwargs)

def loop_over_ensemble_members(data: xr.Dataset, **tqdm_kwargs) -> tqdm:
    """ Loop over ensemble members in `data`, with a progress bar. """
    return loop_over_(data.groupby("number"), unit="member", **tqdm_kwargs)

def loop_over_data_variables(data: xr.Dataset, **tqdm_kwargs) -> tqdm:
    """ Loop over variable keys in `data`, with a progress bar. """
    return loop_over_(data.data_vars.keys(), unit="variable", **tqdm_kwargs)

##### Visualisation

In [5]:
_style_chl = {"cmap": cmc.cm.navia.resampled(15), "norm": LogNorm(vmin=1e-6, vmax=50), "extend": "both"}

styles = {
    "chl": Style(**_style_chl),
}

# Apply general settings
for style in styles.values():
    style.normalize = False

In [6]:
def _add_textbox_to_subplots(text: str, *axs: Iterable[plt.Axes | ekp.Subplot], right=False) -> None:
    """ Add a text box to each of the specified subplots. """
    # Get the plt.Axes for each ekp.Subplot
    axs = [subplot.ax if isinstance(subplot, ekp.Subplot) else subplot for subplot in axs]

    # Set up location
    x = 0.95 if right else 0.05
    horizontalalignment = "right" if right else "left"

    # Add the text
    for ax in axs:
        ax.text(x, 0.95, text, transform=ax.transAxes,
        horizontalalignment=horizontalalignment, verticalalignment="top",
        bbox={"facecolor": "white", "edgecolor": "black", "boxstyle": "round",
              "alpha": 1})

In [7]:
def decorate_fig(fig: ekp.Figure, *, title: Optional[str]="") -> None:
    """ Decorate an earthkit figure with land, coastlines, etc. """
    # Add progress bar because individual steps can be very slow for large plots
    with tqdm(total=4, desc="Decorating", leave=False) as progressbar:
        fig.land()
        progressbar.update()
        fig.coastlines()
        progressbar.update()
        # fig.borders()
        # progressbar.update()
        fig.gridlines(linestyle=plt.rcParams["grid.linestyle"])
        progressbar.update()
        fig.title(title)
        progressbar.update()

(section-download)=
### 2. Download Timeseries Datasets

In [8]:
# Setup
DATASET_ID = "sis-water-level-change-timeseries-cmip6"
YEARS = (1985, 2014) # download present epoch

#### Reanalysis - Present Epoch (1985-2014)

In [9]:
request_reanalysis = {
    "variable": [
        "storm_surge_residual",
    ],
    "experiment": "reanalysis",
    "temporal_aggregation": ["10_min"],
    "year": [f"{year}" for year in range(YEARS[0], YEARS[1]+1)],    
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "version": ["v3"]
}

# Download reanalysis data for present epoch in 1 year batches.
subrequests_reanalysis  = batch_requests(request_reanalysis, n= 1)

data_reanalysis = ekd.from_source("cds", DATASET_ID, *subrequests_reanalysis).to_xarray()

# Display result
data_reanalysis

2026-05-14 11:58:14,374 INFO [2026-05-14T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 19 May. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure/14954).


  0%|          | 0/30 [00:00<?, ?it/s]

<xarray.Dataset> Size: 544GB
Dimensions:               (time: 1577808, stations: 43119)
Coordinates:
  * time                  (time) datetime64[ns] 13MB 1985-01-01 ... 2014-12-3...
  * stations              (stations) uint16 86kB 0 1 2 3 ... 43731 43732 43733
    station_x_coordinate  (stations) float64 345kB dask.array<chunksize=(43119,), meta=np.ndarray>
    station_y_coordinate  (stations) float64 345kB dask.array<chunksize=(43119,), meta=np.ndarray>
Data variables:
    surge                 (time, stations) float64 544GB dask.array<chunksize=(893, 8624), meta=np.ndarray>
Attributes: (12/35)
    Conventions:                   CF-1.6
    featureType:                   timeSeries
    id:                            GTSMv3_surge
    naming_authority:              https://deltares.nl/en
    Metadata_Conventions:          Unidata Dataset Discovery v1.0
    title:                         10-minute timeseries of surge levels
    ...                            ...
    time_coverage_start:           1985-01-01 00:00:00
    time_coverage_end:             1985-01-31 23:50:00
    experiment:                    reanalysis
    date_modified:                 2021-05-06 13:31:31.692854 UTC
    contact:                       Please contact Copernicus User Support on ...
    history:                       This is version 3 of the dataset.

#### Historical - Present Epoch ("cmcc_cm2_vhr4", 1985-2014)

In [10]:
request_historical = {
    "variable": [
        "storm_surge_residual",
    ],
    "experiment": "historical",
    "model": ["cmcc_cm2_vhr4"],
    "temporal_aggregation": ["10_min"],
    "year": [f"{year}" for year in range(YEARS[0], YEARS[1]+1)],    
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "version": ["v1"],
}

# Download historical data
subrequests_historical  = batch_requests(request_historical, n=1)

data_historical = ekd.from_source("cds", DATASET_ID, *subrequests_historical).to_xarray()

2026-05-14 11:58:54,445 INFO [2026-05-14T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 19 May. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure/14954).


  0%|          | 0/30 [00:00<?, ?it/s]

(section-statistics)=
### 3. Calculating Return Periods - (one station - 99th Percentile)

#### 3.1 Declustering extreme surge events

In [13]:
def decluster_surge(data, percentile = 99, station = 22223):

    station_data = data.sel(stations=station)
    time = station_data["time"].values
    surge = station_data["surge"].values

    pXX = np.nanquantile(surge, percentile/100) # find percentile
    pXX_idx = np.where(surge > pXX)[0] # find indices of all storm surge values greater than the threshold value

    min_run_length = np.timedelta64(1, "h") # in datetime format
    max_run_length = np.timedelta64(72, "h") # in datetime format
    
    dt = np.diff(time[pXX_idx]) # find difference in time for all values that exceed threshold
    new_cluster = np.insert(arr =  (dt > min_run_length) , obj =  0, values = True) # insert True in first cluster
    cluster_id = np.cumsum(new_cluster) # cluster all exceedances

    t_pXX = time[pXX_idx] # times that exceed 99 percentile threshold
    
    # find cluster start/end indices
    starts = np.where(new_cluster)[0]
    ends = np.append(starts[1:] - 1, len(t_pXX) - 1 ) # append list of start of next clusters + insert last one    
    # cluster durations
    durations = (t_pXX[ends] - t_pXX[starts]) / np.timedelta64(1, "h") # convert nanoseconds to hours

    declustered_idx = []
    
    for s, e, d in zip(starts, ends, durations):
        if d <= 72: # max storm duration
            i = pXX_idx[s:e+1] # slice each cluster 
            declustered_idx.append(i[np.argmax(surge[i])]) # find maximum surge value index in each cluster
    
    declustered_surge = surge[declustered_idx] 

    return declustered_surge

declustered_reanalysis_surge = decluster_surge(data_reanalysis) # takes about ~ 5 mins to run.
# declustered_historical_surge = decluster_surge(data_historical)

#### 3.2 Fitting General Pareto Distribution (GPD) to declustered surge extremes

In [49]:
from scipy.stats import genpareto

rean_shape_fit, rean_loc_fit, rean_scale_fit = genpareto.fit(declustered_reanalysis_surge, fc = 0.005) # fit general pareto distribution 
# hist_shape_fit, hist_loc_fit, hist_scale_fit = genpareto.fit(declustered_historical_surge, 0) # fit general pareto distribution 

#### 3.3 Calculating surge level at multiple return periods

In [50]:
def compute_lambda(time, declustered_surge):
    n_events = len(declustered_surge)
    
    duration_days = (time[-1] - time[0]) / np.timedelta64(1, "D")
    duration_years = duration_days / 365.25
    
    return n_events / duration_years

lambda_rean = compute_lambda(data_reanalysis["time"].values, declustered_reanalysis_surge)

In [51]:
return_periods = [1, 2, 5, 10, 25, 50, 75, 100]

rp_rean_data = {
    "Return period (years)": return_periods,
    "Surge level (m)": [
        genpareto.isf(
            1 / (rp * lambda_rean),
            c=rean_shape_fit,
            loc=rean_loc_fit,
            scale=rean_scale_fit
        )
        for rp in return_periods
    ]
}

pd.DataFrame(rp_rean_data)

,Return period (years),Surge level (m)
0,1,0.827447
1,2,0.901265
2,5,0.999239
3,10,1.073653
4,25,1.172420
5,50,1.247435
6,75,1.291436
7,100,1.322710


(section-results)=
### 4. Results

#### Comparing return period with CDS downloads

In [17]:
# Setup
DATASET_ID = "sis-water-level-change-indicators-cmip6"

In [ ]:
request_indicator_reanalysis = {
    "variable": ["surge_level"],
    "derived_variable": ["absolute_value"],
    "product_type": ["reanalysis"],
    "confidence_interval": ["best_fit"],
    "period": ["1985_2014"]
}

request_indicator_reanalysis_1_yr = request_indicator_reanalysis | {"statistic": ["1_year"]}
request_indicator_reanalysis_2_yr = request_indicator_reanalysis | {"statistic": ["2_year"]}
request_indicator_reanalysis_5_yr = request_indicator_reanalysis | {"statistic": ["5_year"]}
request_indicator_reanalysis_10_yr = request_indicator_reanalysis | {"statistic": ["10_year"]}
request_indicator_reanalysis_25_yr = request_indicator_reanalysis | {"statistic": ["25_year"]}
request_indicator_reanalysis_50_yr = request_indicator_reanalysis | {"statistic": ["50_year"]}
request_indicator_reanalysis_100_yr = request_indicator_reanalysis | {"statistic": ["100_year"]}


data_rp_1 = ekd.from_source("cds", DATASET_ID, request_indicator_reanalysis_1_yr).to_xarray()
data_rp_2 = ekd.from_source("cds", DATASET_ID, request_indicator_reanalysis_2_yr).to_xarray()
data_rp_5 = ekd.from_source("cds", DATASET_ID, request_indicator_reanalysis_5_yr).to_xarray()
data_rp_10 = ekd.from_source("cds", DATASET_ID, request_indicator_reanalysis_10_yr).to_xarray()
data_rp_25 = ekd.from_source("cds", DATASET_ID, request_indicator_reanalysis_25_yr).to_xarray()
data_rp_50 = ekd.from_source("cds", DATASET_ID, request_indicator_reanalysis_50_yr).to_xarray()
data_rp_100 = ekd.from_source("cds", DATASET_ID, request_indicator_reanalysis_100_yr).to_xarray()

In [58]:
data_rp_1["return_mean_surge_level"].sel(stations=22223).values

array(0.848)

In [59]:
data_rp_10["return_mean_surge_level"].sel(stations=22223).values

array(1.192)

#### Comparing return period between historical projection and reanalysis.

In [ ]:
# collapsable code cell

# code is included for transparency but also learning purposes and gives users the chance to adapt the code used for the assessment as they wish

## ℹ️ If you want to know more

### Key resources

List some key resources related to this assessment. E.g. CDS entries, applications, dataset documentation, external pages.
Also list any code libraries used (if applicable).

Code libraries used:
* 

### References

List the references used in the Notebook here.

E.g.

[[1]](https://doi.org/10.1038/s41598-018-20628-2) Rodriguez, D., De Voil, P., Hudson, D., Brown, J. N., Hayman, P., Marrou, H., & Meinke, H. (2018). Predicting optimum crop designs using crop models and seasonal climate forecasts. Scientific reports, 8(1), 2231.